In [ ]:
# ── 1. Imports ───────────────────────────────────────────────
import random
import re
from collections import defaultdict

random.seed(42)  # reproducible outputs

In [ ]:
# ── 2. Sample Corpus ─────────────────────────────────────────
corpus = """
To be or not to be that is the question whether tis nobler in the mind
to suffer the slings and arrows of outrageous fortune or to take arms
against a sea of troubles and by opposing end them to die to sleep no more
and by a sleep to say we end the heartache and the thousand natural shocks
that flesh is heir to tis a consummation devoutly to be wished to die to sleep
to sleep perchance to dream in that sleep of death what dreams may come
when we have shuffled off this mortal coil must give us pause there is the respect
that makes calamity of so long life for who would bear the whips and scorns of time
the oppressors wrong the proud mans contumely the pangs of despised love
the laws delay the insolence of office and the spurns that patient merit
of the unworthy takes when he himself might his quietus make with a bare bodkin
"""


In [ ]:
# ── 3. Tokenizers ─────────────────────────────────────────────
def word_tokenize(text):
    """Lowercase and extract word tokens."""
    return re.findall(r"[\w']+|[.,!?;]", text.lower())

def char_tokenize(text):
    """Split into characters, keep spaces."""
    return list(text.lower())

In [ ]:
# ── 4. Build Markov Chain ─────────────────────────────────────
def build_chain(tokens, order=2):
    """
    Build a transition table.
    chain[state] = {next_token: count, ...}
    state is a tuple of `order` tokens.

    Loop range: range(len(tokens) - order)
    → i goes from 0 to len(tokens) - order - 1
    → tokens[i + order] is always valid. No off-by-one.
    """
    chain = defaultdict(lambda: defaultdict(int))
    for i in range(len(tokens) - order):
        state = tuple(tokens[i : i + order])
        next_token = tokens[i + order]
        chain[state][next_token] += 1
    return chain

In [ ]:
# ── 5. Weighted Random Pick ───────────────────────────────────
def weighted_pick(choices: dict):
    """Pick a key from {token: count} weighted by frequency."""
    tokens = list(choices.keys())
    weights = list(choices.values())
    return random.choices(tokens, weights=weights, k=1)[0]

In [ ]:
# ── 6. Generate Text (fixed dead-end handling) ───────────────
def generate_text(chain, order, length=80, seed=None):
    """
    Generate text using the Markov chain.

    Fix: on dead end, pick a new random state and continue
    WITHOUT re-appending the full state to result.
    Previously result.extend(list(state)) caused visible
    repeated chunks in the output.

    Parameters
    ----------
    chain   : dict  — transition table from build_chain()
    order   : int   — n-gram order used to build the chain
    length  : int   — number of tokens to generate
    seed    : tuple — optional starting state (tuple of `order` tokens)

    Returns
    -------
    list of tokens
    """
    keys = list(chain.keys())
    state = seed if (seed and seed in chain) else random.choice(keys)

    result = list(state)

    for _ in range(length - order):
        nexts = chain.get(state)
        if not nexts:
            # Dead end: jump to a new random state, don't duplicate output
            state = random.choice(keys)
            continue

        next_token = weighted_pick(nexts)
        result.append(next_token)
        state = tuple(result[-order:])

    return result

In [ ]:
# ── 7. Format Output (fixed punctuation spacing) ─────────────
def tokens_to_text(tokens, mode='word'):
    """
    Join tokens into a string.

    Fix: word mode now removes spaces before punctuation.
    Previously ' '.join(tokens) produced "question , tis"
    instead of "question, tis".
    """
    if mode == 'char':
        return ''.join(tokens)

    # word mode: strip space before punctuation marks
    text = ' '.join(tokens)
    text = re.sub(r"\s+([.,!?;:])", r"\1", text)
    return text

In [ ]:
# ── 8. Display Chain Stats ────────────────────────────────────
def print_chain_stats(chain, tokens, order, mode):
    print("=" * 55)
    print("         MARKOV CHAIN STATISTICS")
    print("=" * 55)
    print(f"  Mode          : {mode}")
    print(f"  Order (N)     : {order}")
    print(f"  Total tokens  : {len(tokens)}")
    print(f"  Unique states : {len(chain)}")
    print("=" * 55)

def print_chain_sample(chain, n=10):
    print("\n── Transition Table Sample ──────────────────────────")
    for i, (state, nexts) in enumerate(chain.items()):
        if i >= n:
            break
        top = sorted(nexts.items(), key=lambda x: -x[1])[:4]
        top_str = ', '.join(f"'{t}'({c})" for t, c in top)
        print(f"  {state}  →  {top_str}")
    print()

In [ ]:
# ── 9. Run: Word-level Example ────────────────────────────────
print("\n" + "━" * 55)
print("  WORD-LEVEL MARKOV CHAIN  (order=2)")
print("━" * 55)

ORDER = 2
MODE  = 'word'

tokens_word = word_tokenize(corpus)
chain_word  = build_chain(tokens_word, order=ORDER)

print_chain_stats(chain_word, tokens_word, ORDER, MODE)
print_chain_sample(chain_word, n=8)

generated_tokens = generate_text(chain_word, order=ORDER, length=60)
generated_text   = tokens_to_text(generated_tokens, mode=MODE)

print("── Generated Text ───────────────────────────────────")
print(generated_text)

In [ ]:
# ── 10. Run: Character-level Example ─────────────────────────
print("\n\n" + "━" * 55)
print("  CHARACTER-LEVEL MARKOV CHAIN  (order=4)")
print("━" * 55)

ORDER_C = 4
MODE_C  = 'char'

tokens_char = char_tokenize(corpus)
chain_char  = build_chain(tokens_char, order=ORDER_C)

print_chain_stats(chain_char, tokens_char, ORDER_C, MODE_C)
print_chain_sample(chain_char, n=8)

generated_chars     = generate_text(chain_char, order=ORDER_C, length=200)
generated_char_text = tokens_to_text(generated_chars, mode=MODE_C)

print("── Generated Text ───────────────────────────────────")
print(generated_char_text)

In [ ]:
# ── 11. Comparison: Order 1 vs 2 vs 3 ────────────────────────
print("\n\n" + "━" * 55)
print("  COMPARISON: Order 1 vs 2 vs 3  (word-level)")
print("━" * 55)

for order in [1, 2, 3]:
    ch   = build_chain(tokens_word, order=order)
    toks = generate_text(ch, order=order, length=40)
    text = tokens_to_text(toks, mode='word')
    print(f"\n  Order {order}:")
    print(f"  {text}")

In [ ]:
# ── 12. Custom Corpus ─────────────────────────────────────────
print("\n\n" + "━" * 55)
print("  CUSTOM CORPUS EXAMPLE")
print("━" * 55)

custom_corpus = """
The quick brown fox jumps over the lazy dog.
The dog barked at the fox. The fox ran away quickly.
A quick fox and a lazy dog lived near the brown hill.
The hill was covered with quick brown grass and lazy flowers.
"""

custom_tokens = word_tokenize(custom_corpus)
custom_chain  = build_chain(custom_tokens, order=2)
custom_output = generate_text(custom_chain, order=2, length=30)

print("Input corpus word count :", len(custom_tokens))
print("Generated:")
print(tokens_to_text(custom_output, mode='word'))